In [ ]:
# === ALL-IN-ONE: arrays-only (no state dicts in function signatures) Shift-NMFk-TFGM v12 + sample ===
from __future__ import annotations
import numpy as np
from typing import Optional, Tuple, Literal, List, Dict

# --------------------------
# Shift operators & helpers
# --------------------------
def circ_shift_linpos_1d(x: np.ndarray, tau: float) -> np.ndarray:
    """Circular, linear-interp shift by real-valued tau (nonnegative output)."""
    L = int(x.shape[0])
    if L == 0: return x.copy()
    k = int(np.floor(tau))
    a = float(tau - k)
    idx0 = (np.arange(L) - k) % L
    idx1 = (idx0 - 1) % L
    return (1.0 - a) * x[idx0] + a * x[idx1]

def vec_outer(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return np.outer(a, b).reshape(-1)

def xcorr_circ(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    A = np.fft.fft(a)
    B = np.fft.fft(b)
    return np.real(np.fft.ifft(np.conj(A)*B))

def argmax_local_circ(c: np.ndarray, center: float, window: int) -> float:
    L = c.size
    k0 = int(np.round(np.mod(center, L)))
    idx = (k0 + np.arange(-window, window+1)) % L
    j = idx[int(np.argmax(c[idx]))]
    jm1, jp1 = (j-1) % L, (j+1) % L
    y_m1, y0, y_p1 = c[jm1], c[j], c[jp1]
    denom = (y_m1 - 2*y0 + y_p1)
    delta = 0.0 if abs(denom) < 1e-12 else 0.5*(y_m1 - y_p1)/denom
    tau = (j + delta) % L
    if tau > L/2: tau -= L
    return float(tau)

# --------------------------
# Forward models / utilities
# --------------------------
def reconstruct_from_params(W: np.ndarray, F: np.ndarray, S: np.ndarray,
                            tau_f: np.ndarray, tau_t: np.ndarray) -> np.ndarray:
    """X_hat[n,f,t] = sum_i W[n,i] * shift(F[i], tau_f[n,i]) ⊗ shift(S[i], tau_t[n,i])."""
    N, K = W.shape
    Fdim, Tdim = F.shape[1], S.shape[1]
    X_hat = np.zeros((N, Fdim, Tdim), dtype=float)
    for n in range(N):
        for i in range(K):
            Fi = circ_shift_linpos_1d(F[i], float(tau_f[n,i]))
            Si = circ_shift_linpos_1d(S[i], float(tau_t[n,i]))
            X_hat[n] += W[n,i] * np.outer(Fi, Si)
    return X_hat

def compute_shifted(F: np.ndarray, S: np.ndarray, tau_f: np.ndarray, tau_t: np.ndarray
                    ) -> Tuple[np.ndarray, np.ndarray]:
    """Return per-sensor shifted dictionaries: Fi_shifted[n,i,f], Si_shifted[n,i,t]."""
    N, K = tau_f.shape
    Fdim, Tdim = F.shape[1], S.shape[1]
    Fi_shifted = np.zeros((N, K, Fdim))
    Si_shifted = np.zeros((N, K, Tdim))
    for n in range(N):
        for i in range(K):
            Fi_shifted[n,i] = circ_shift_linpos_1d(F[i], tau_f[n,i])
            Si_shifted[n,i] = circ_shift_linpos_1d(S[i], tau_t[n,i])
    return Fi_shifted, Si_shifted

def reconstruct_from_shifted(W: np.ndarray, Fi_shifted: np.ndarray, Si_shifted: np.ndarray) -> np.ndarray:
    N = W.shape[0]
    Fdim = Fi_shifted.shape[2]
    Tdim = Si_shifted.shape[2]
    X_hat = np.zeros((N, Fdim, Tdim), dtype=float)
    for n in range(N):
        X_hat[n] = np.tensordot(W[n], np.einsum("if,it->ift", Fi_shifted[n], Si_shifted[n]), axes=(0,0))
    return X_hat

def relerr(A: np.ndarray, B: np.ndarray) -> float:
    return float(np.linalg.norm(A-B)/(np.linalg.norm(A)+1e-12))

def compute_loss(X: np.ndarray, W: np.ndarray, F: np.ndarray, S: np.ndarray,
                 tau_f: np.ndarray, tau_t: np.ndarray) -> Tuple[float, float]:
    Fdim, Tdim = X.shape[1], X.shape[2]
    X_hat = reconstruct_from_params(W, F, S, tau_f, tau_t)
    loss = 0.5*np.sum((X - X_hat)**2)/(Fdim*Tdim)
    R = relerr(X, X_hat)
    return float(loss), float(R)

# --------------------------
# W update (ridge LS, ≥0)
# --------------------------
def update_W_ls_from_shifted(X: np.ndarray, W: np.ndarray, Fi_shifted: np.ndarray, Si_shifted: np.ndarray,
                             ridge_w: float = 0.0) -> np.ndarray:
    """Return updated W via ridge least squares (per sensor, nonnegative)."""
    N, Fdim, Tdim = X.shape
    FT = Fdim*Tdim
    K = W.shape[1]
    W_new = np.empty_like(W)
    for n in range(N):
        A = np.stack([vec_outer(Fi_shifted[n,i], Si_shifted[n,i]) for i in range(K)], axis=1)  # (FT,K)
        b = X[n].reshape(FT)
        AtA = A.T @ A
        if ridge_w > 0:
            AtA = AtA + ridge_w * np.eye(K)
        Atb = A.T @ b
        w = np.linalg.solve(AtA, Atb)
        W_new[n] = np.clip(w, 1e-12, None)
    return W_new

# --------------------------
# (F,S) update: SVD rank-1 with accept-if-better
# --------------------------
def update_rank1_for_component(X: np.ndarray, W: np.ndarray, F: np.ndarray, S: np.ndarray,
                               tau_f: np.ndarray, tau_t: np.ndarray, i: int
                               ) -> Tuple[np.ndarray, np.ndarray]:
    """Return (F_updated, S_updated) with only component i potentially modified."""
    N, K = W.shape
    Fdim, Tdim = F.shape[1], S.shape[1]

    # Shift once
    Fi_shifted, Si_shifted = compute_shifted(F, S, tau_f, tau_t)

    # Residual excluding i
    R_i = np.zeros((N, Fdim, Tdim))
    for n in range(N):
        recon_wo_i = np.zeros((Fdim, Tdim))
        for j in range(K):
            if j == i: continue
            recon_wo_i += W[n,j] * np.outer(Fi_shifted[n,j], Si_shifted[n,j])
        R_i[n] = X[n] - recon_wo_i

    # Weighted aligned residual
    w = W[:, i]
    wsum = float(np.sum(w))
    if wsum <= 1e-16:
        return F, S
    A = np.zeros((Fdim, Tdim))
    for n in range(N):
        if w[n] <= 0: continue
        Rn = R_i[n]
        if tau_t[n,i] != 0.0:
            Rn = np.stack([circ_shift_linpos_1d(Rn[r], -tau_t[n,i]) for r in range(Fdim)], axis=0)
        if tau_f[n,i] != 0.0:
            Rn = np.stack([circ_shift_linpos_1d(Rn[:,c], -tau_f[n,i]) for c in range(Tdim)], axis=1)
        A += w[n] * Rn
    A /= wsum

    # Top SVD, project nonneg, keep amplitude via sqrt(s1)
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    u = np.maximum(U[:,0], 1e-12)
    v = np.maximum(Vt[0,:], 1e-12)
    s1 = float(max(s[0], 0.0))
    scale = np.sqrt(s1)
    Fi_new = u * scale
    Si_new = v * scale

    F_old_i = F[i].copy()
    S_old_i = S[i].copy()
    F_cand = F.copy()
    S_cand = S.copy()
    F_cand[i] = Fi_new
    S_cand[i] = Si_new

    loss_old, _ = compute_loss(X, W, F, S, tau_f, tau_t)
    loss_new, _ = compute_loss(X, W, F_cand, S_cand, tau_f, tau_t)
    if loss_new <= loss_old + 1e-12:
        return F_cand, S_cand
    else: # rollback
        F_rollback = F.copy()
        S_rollback = S.copy()
        F_rollback[i] = F_old_i
        S_rollback[i] = S_old_i
        return F_rollback, S_rollback

# --------------------------
# Snapping of shifts
# --------------------------
def snap_shifts(X: np.ndarray, W: np.ndarray, F: np.ndarray, S: np.ndarray,
                tau_f: np.ndarray, tau_t: np.ndarray,
                snap_window_f: int = 6, snap_window_t: int = 6
                ) -> Tuple[np.ndarray, np.ndarray]:
    """Return (tau_f_new, tau_t_new) snapped via local xcorr around current values."""
    N, K = W.shape
    Fdim, Tdim = F.shape[1], S.shape[1]
    Fi_shifted, Si_shifted = compute_shifted(F, S, tau_f, tau_t)
    full = reconstruct_from_shifted(W, Fi_shifted, Si_shifted)

    tau_f_new = tau_f.copy()
    tau_t_new = tau_t.copy()
    for n in range(N):
        for i in range(K):
            R_i = X[n] - (full[n] - W[n,i]*np.outer(Fi_shifted[n,i], Si_shifted[n,i]))
            # freq snap
            w_t = Si_shifted[n,i] / (np.sum(Si_shifted[n,i]) + 1e-12)
            freq_prof = R_i @ w_t
            c = xcorr_circ(F[i], freq_prof)
            tau_f_new[n,i] = argmax_local_circ(c, tau_f[n,i], snap_window_f)
            # time snap
            w_f = Fi_shifted[n,i] / (np.sum(Fi_shifted[n,i]) + 1e-12)
            time_prof = R_i.T @ w_f
            c_t = xcorr_circ(S[i], time_prof)
            tau_t_new[n,i] = argmax_local_circ(c_t, tau_t[n,i], snap_window_t)
    return tau_f_new, tau_t_new

# --------------------------
# Fit loops (arrays only)
# --------------------------
def _log_step(X: np.ndarray, W: np.ndarray, F: np.ndarray, S: np.ndarray,
              tau_f: np.ndarray, tau_t: np.ndarray,
              it: int, tag: str, verbose: bool, loss_history: List[float]) -> None:
    loss, R = compute_loss(X, W, F, S, tau_f, tau_t)
    loss_history.append(loss)
    if verbose and (it % 10 == 0 or it == 1):
        print(f"[{tag:>6} {it:4d}] loss={loss:.6e}  R={R:.6f}")

def fit_monolithic(X: np.ndarray, W: np.ndarray, F: np.ndarray, S: np.ndarray,
                   tau_f: np.ndarray, tau_t: np.ndarray,
                   n_iters: int = 400, freeze_tau_iters: int = 120, snap_every: int = 5,
                   snap_window_f: int = 6, snap_window_t: int = 6,
                   inner_rank1_iters: int = 1, ridge_w: float = 0.0, verbose: bool = True
                   ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[float]]:
    loss_history: List[float] = []
    for it in range(1, n_iters+1):
        Fi, Si = compute_shifted(F, S, tau_f, tau_t)
        W = update_W_ls_from_shifted(X, W, Fi, Si, ridge_w=ridge_w)
        for i in range(F.shape[0]):
            for _ in range(inner_rank1_iters):
                F, S = update_rank1_for_component(X, W, F, S, tau_f, tau_t, i)
        if it > freeze_tau_iters and snap_every > 0 and (it % snap_every == 0):
            tau_f, tau_t = snap_shifts(X, W, F, S, tau_f, tau_t,
                                       snap_window_f=snap_window_f, snap_window_t=snap_window_t)
        _log_step(X, W, F, S, tau_f, tau_t, it, "ALL", verbose, loss_history)
    return W, F, S, tau_f, tau_t, loss_history

def fit_shifts_only(X: np.ndarray, W: np.ndarray, F: np.ndarray, S: np.ndarray,
                    tau_f: np.ndarray, tau_t: np.ndarray,
                    iters_w: int = 60, iters_snap: int = 80,
                    snap_window_f: int = 6, snap_window_t: int = 6,
                    ridge_w: float = 0.0, verbose: bool = True
                    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[float]]:
    loss_history: List[float] = []
    for it in range(1, iters_w+1):
        Fi, Si = compute_shifted(F, S, tau_f, tau_t)
        W = update_W_ls_from_shifted(X, W, Fi, Si, ridge_w=ridge_w)
        _log_step(X, W, F, S, tau_f, tau_t, it, "W", verbose, loss_history)
    for it in range(1, iters_snap+1):
        Fi, Si = compute_shifted(F, S, tau_f, tau_t)
        W = update_W_ls_from_shifted(X, W, Fi, Si, ridge_w=ridge_w)
        tau_f, tau_t = snap_shifts(X, W, F, S, tau_f, tau_t,
                                   snap_window_f=snap_window_f, snap_window_t=snap_window_t)
        _log_step(X, W, F, S, tau_f, tau_t, it, "W+τ", verbose, loss_history)
    return W, F, S, tau_f, tau_t, loss_history

def fit_staged(X: np.ndarray, W: np.ndarray, F: np.ndarray, S: np.ndarray,
               tau_f: np.ndarray, tau_t: np.ndarray,
               iters_w: int = 60, iters_fs: int = 120, iters_snap: int = 80,
               inner_rank1_iters: int = 1,
               snap_window_f: int = 6, snap_window_t: int = 6,
               ridge_w: float = 0.0, verbose: bool = True
               ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[float]]:
    loss_history: List[float] = []
    # Stage 1: W
    for it in range(1, iters_w+1):
        Fi, Si = compute_shifted(F, S, tau_f, tau_t)
        W = update_W_ls_from_shifted(X, W, Fi, Si, ridge_w=ridge_w)
        _log_step(X, W, F, S, tau_f, tau_t, it, "W", verbose, loss_history)
    # Stage 2: W + (F,S)
    for it in range(1, iters_fs+1):
        Fi, Si = compute_shifted(F, S, tau_f, tau_t)
        W = update_W_ls_from_shifted(X, W, Fi, Si, ridge_w=ridge_w)
        for i in range(F.shape[0]):
            for _ in range(inner_rank1_iters):
                F, S = update_rank1_for_component(X, W, F, S, tau_f, tau_t, i)
        _log_step(X, W, F, S, tau_f, tau_t, it, "W+FS", verbose, loss_history)
    # Stage 3: W + (F,S) + snap
    for it in range(1, iters_snap+1):
        Fi, Si = compute_shifted(F, S, tau_f, tau_t)
        W = update_W_ls_from_shifted(X, W, Fi, Si, ridge_w=ridge_w)
        for i in range(F.shape[0]):
            for _ in range(inner_rank1_iters):
                F, S = update_rank1_for_component(X, W, F, S, tau_f, tau_t, i)
        tau_f, tau_t = snap_shifts(X, W, F, S, tau_f, tau_t,
                                   snap_window_f=snap_window_f, snap_window_t=snap_window_t)
        _log_step(X, W, F, S, tau_f, tau_t, it, "ALL", verbose, loss_history)
    return W, F, S, tau_f, tau_t, loss_history

def prepare_initial_params(X: np.ndarray, K: int,
                           W0: Optional[np.ndarray] = None, F0: Optional[np.ndarray] = None, S0: Optional[np.ndarray] = None,
                           tau_f0: Optional[np.ndarray] = None, tau_t0: Optional[np.ndarray] = None,
                           seed: Optional[int] = 0
                           ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    assert X.ndim == 3 and np.all(X >= 0), "X must be nonnegative with shape (N,F,T)"
    N, Fdim, Tdim = X.shape
    rng = np.random.default_rng(seed)
    W = np.maximum(1e-3 + 0.1*rng.random((N,K)), 1e-8) if W0 is None else W0.copy()
    F = np.maximum(1e-3 + 0.1*rng.random((K,Fdim)), 1e-8) if F0 is None else F0.copy()
    S = np.maximum(1e-3 + 0.1*rng.random((K,Tdim)), 1e-8) if S0 is None else S0.copy()
    tau_f = np.zeros((N,K)) if tau_f0 is None else tau_f0.copy()
    tau_t = np.zeros((N,K)) if tau_t0 is None else tau_t0.copy()
    return W, F, S, tau_f, tau_t

def fit(X: np.ndarray, K: int,
        mode: Literal["monolithic","staged","shifts_only"] = "monolithic",
        seed: Optional[int] = 0,
        # common knobs
        ridge_w: float = 0.0, verbose: bool = True,
        # monolithic
        n_iters: int = 400, freeze_tau_iters: int = 120, snap_every: int = 5,
        snap_window_f: int = 6, snap_window_t: int = 6, inner_rank1_iters: int = 1,
        # staged/shifts-only
        iters_w: int = 60, iters_fs: int = 120, iters_snap: int = 80,
        # explicit init (arrays)
        W0: Optional[np.ndarray] = None, F0: Optional[np.ndarray] = None, S0: Optional[np.ndarray] = None,
        tau_f0: Optional[np.ndarray] = None, tau_t0: Optional[np.ndarray] = None
        ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[float]]:
    W, F, S, tau_f, tau_t = prepare_initial_params(X, K, W0, F0, S0, tau_f0, tau_t0, seed=seed)
    if mode == "shifts_only":
        W, F, S, tau_f, tau_t, hist = fit_shifts_only(
            X, W, F, S, tau_f, tau_t,
            iters_w=iters_w, iters_snap=iters_snap,
            snap_window_f=snap_window_f, snap_window_t=snap_window_t,
            ridge_w=ridge_w, verbose=verbose
        )
    elif mode == "staged":
        W, F, S, tau_f, tau_t, hist = fit_staged(
            X, W, F, S, tau_f, tau_t,
            iters_w=iters_w, iters_fs=iters_fs, iters_snap=iters_snap,
            inner_rank1_iters=inner_rank1_iters,
            snap_window_f=snap_window_f, snap_window_t=snap_window_t,
            ridge_w=ridge_w, verbose=verbose
        )
    else:
        W, F, S, tau_f, tau_t, hist = fit_monolithic(
            X, W, F, S, tau_f, tau_t,
            n_iters=n_iters, freeze_tau_iters=freeze_tau_iters, snap_every=snap_every,
            snap_window_f=snap_window_f, snap_window_t=snap_window_t,
            inner_rank1_iters=inner_rank1_iters, ridge_w=ridge_w, verbose=verbose
        )
    return W, F, S, tau_f, tau_t, hist

# --------------------------
# Convenience: xcorr seeding
# --------------------------
def seed_shifts_xcorr(X: np.ndarray, F0: np.ndarray, S0: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    N,F,T = X.shape
    K = F0.shape[0]
    tau_f0 = np.zeros((N,K))
    tau_t0 = np.zeros((N,K))
    def xcorr_tau(a,b):
        A = np.fft.fft(a)
        B = np.fft.fft(b)
        c = np.real(np.fft.ifft(np.conj(A)*B))
        k = int(np.argmax(c))
        km1=(k-1)%c.size
        kp1=(k+1)%c.size
        denom = (c[km1]-2*c[k]+c[kp1])
        delta = 0.0 if abs(denom)<1e-12 else 0.5*(c[km1]-c[kp1])/denom
        tau = (k+delta) % c.size
        if tau > c.size/2: tau -= c.size
        return float(tau)
    for n in range(N):
        for i in range(K):
            w_t = S0[i]/(S0[i].sum()+1e-12)
            w_f = F0[i]/(F0[i].sum()+1e-12)
            freq_profile = X[n] @ w_t
            time_profile = X[n].T @ w_f
            tau_f0[n,i] = xcorr_tau(F0[i], freq_profile)
            tau_t0[n,i] = xcorr_tau(S0[i], time_profile)
    return tau_f0, tau_t0

# --------------------------
# Utilities: noise, Doppler, triangulation, CRB, bootstrap
# --------------------------
def measure_snr_db(X_clean: np.ndarray, X_noisy: np.ndarray) -> float:
    num = np.linalg.norm(X_clean)
    den = np.linalg.norm(X_noisy - X_clean) + 1e-12
    return float(20.0 * np.log10(num / den))

def add_noise_by_snr_clipped_per_sensor(X_clean: np.ndarray, snr_db: float = 30.0,
                                        rng: Optional[np.random.Generator] = None, max_iter: int = 40) -> np.ndarray:
    rng = np.random.default_rng(0) if rng is None else rng
    X_noisy = np.empty_like(X_clean)
    for n in range(X_clean.shape[0]):
        noise = rng.normal(0.0, 1.0, size=X_clean[n].shape)
        target = (np.linalg.norm(X_clean[n]) + 1e-12) * (10.0 ** (-snr_db / 20.0))
        lo, hi = 0.0, (np.linalg.norm(X_clean[n]) / (np.linalg.norm(noise) + 1e-12)) * 10
        for _ in range(max_iter):
            a = 0.5*(lo+hi)
            X_try = np.clip(X_clean[n] + a*noise, 0.0, None)
            err = np.linalg.norm(X_try - X_clean[n])
            if err > target: hi = a
            else: lo = a
        X_noisy[n] = np.clip(X_clean[n] + lo*noise, 0.0, None)
    return X_noisy

def unwrap_bins(tau_bins: np.ndarray, F: int) -> np.ndarray:
    return ((tau_bins + F/2) % F) - F/2

def unwrap_frames(tau_frames: np.ndarray, T: int) -> np.ndarray:
    return ((tau_frames + T/2) % T) - T/2

def tau_to_doppler_hz(tau_f_bins: np.ndarray, bin_hz: float) -> np.ndarray:
    return tau_f_bins * float(bin_hz)

# ---- TDOA trilateration via Gauss–Newton ----
def tdoa_residual_and_jac(s: np.ndarray, P: np.ndarray, dt: np.ndarray, c: float, ref: int = 0) -> Tuple[np.ndarray, np.ndarray]:
    r_list=[]
    J_list=[]
    pr = P[ref]
    for n in range(P.shape[0]):
        if n == ref: continue
        pn = P[n]
        vn = s - pn
        rn = np.linalg.norm(vn) + 1e-12
        vr = s - pr
        rr = np.linalg.norm(vr) + 1e-12
        r = rn - rr - c*dt[n]
        J = (vn/rn) - (vr/rr)
        r_list.append(r)
        J_list.append(J)
    return np.array(r_list), np.vstack(J_list)

def tdoa_gn(P: np.ndarray, dt: np.ndarray, c: float, ref: int = 0,
            x0: Optional[np.ndarray] = None, max_iter: int = 100, tol: float = 1e-10) -> Tuple[np.ndarray, Dict[str, object]]:
    if x0 is None: x0 = P.mean(axis=0).copy()
    s = x0.copy()
    for it in range(max_iter):
        r, J = tdoa_residual_and_jac(s, P, dt, c, ref=ref)
        H = J.T @ J
        g = J.T @ r
        try: step = np.linalg.solve(H, -g)
        except np.linalg.LinAlgError: step = -np.linalg.pinv(H) @ g
        s_new = s + step
        if np.linalg.norm(step) < tol: s = s_new; break
        s = s_new
    r, J = tdoa_residual_and_jac(s, P, dt, c, ref=ref)
    try: cov = np.linalg.inv(J.T @ J)
    except np.linalg.LinAlgError: cov = np.linalg.pinv(J.T @ J)
    return s, {"iters": it+1, "res_norm": float(np.linalg.norm(r)), "cov_unit": cov}

def crb_covariance(P: np.ndarray, dt: np.ndarray, c: float, ref: int = 0,
                   sigma_tdoa_sec: Optional[np.ndarray] = None, s_eval: Optional[np.ndarray] = None):
    if sigma_tdoa_sec is None: 
        return None
    N = P.shape[0]
    if np.isscalar(sigma_tdoa_sec):
        sig = np.full(N, float(sigma_tdoa_sec))
    else:
        sig = np.array(sigma_tdoa_sec, dtype=float).reshape(-1)
        assert sig.size == N, "sigma_tdoa_sec must be scalar or length N"
    if s_eval is None:
        s_eval, _ = tdoa_gn(P, dt, c, ref=ref)
    r, J = tdoa_residual_and_jac(s_eval, P, dt, c, ref=ref)
    var_r = []
    for n in range(N):
        if n == ref: continue
        var_r.append((c**2) * (sig[n]**2 + sig[ref]**2))
    Sigma_r = np.diag(var_r)
    try:
        cov = np.linalg.inv(J.T @ np.linalg.inv(Sigma_r) @ J)
    except np.linalg.LinAlgError:
        cov = np.linalg.pinv(J.T @ np.linalg.inv(Sigma_r) @ J)
    return cov

# ---- Bootstrap triangulation (arrays-only API) ----
def reestimate_tau_t_shifts_only(Xb: np.ndarray, W_seed: np.ndarray, F_seed: np.ndarray, S_seed: np.ndarray,
                                 tau_f_seed: np.ndarray, tau_t_seed: np.ndarray,
                                 iters_w: int = 30, iters_snap: int = 30) -> np.ndarray:
    K = F_seed.shape[0]
    Wb, Fb, Sb, taufb, tautb, _ = fit(
        Xb, K, mode="shifts_only",
        iters_w=iters_w, iters_snap=iters_snap,
        snap_window_f=3, snap_window_t=3,
        ridge_w=0.0, verbose=False,
        W0=W_seed, F0=F_seed, S0=S_seed, tau_f0=tau_f_seed, tau_t0=tau_t_seed
    )
    return tautb

def reestimate_tau_f_shifts_only(Xb: np.ndarray, W_seed: np.ndarray, F_seed: np.ndarray, S_seed: np.ndarray,
                                 tau_f_seed: np.ndarray, tau_t_seed: np.ndarray,
                                 iters_w: int = 20, iters_snap: int = 20) -> np.ndarray:
    K = F_seed.shape[0]
    Wb, Fb, Sb, taufb, tautb, _ = fit(
        Xb, K, mode="shifts_only",
        iters_w=iters_w, iters_snap=iters_snap,
        snap_window_f=3, snap_window_t=3,
        ridge_w=0.0, verbose=False,
        W0=W_seed, F0=F_seed, S0=S_seed, tau_f0=tau_f_seed, tau_t0=tau_t_seed
    )
    return taufb

def bootstrap_triangulation(W: np.ndarray, F: np.ndarray, S: np.ndarray, tau_f: np.ndarray, tau_t: np.ndarray,
                            P: np.ndarray, hop_sec: float, c: float, ref: int = 0,
                            B: int = 200, rng: Optional[np.random.Generator] = None,
                            snr_db: float = 30.0, base_X: Optional[np.ndarray] = None):
    rng = np.random.default_rng(0) if rng is None else rng
    N, K = W.shape
    Tframes = S.shape[1]

    tau_unw = unwrap_frames(tau_t, Tframes)
    dt_base = (tau_unw - tau_unw[ref:ref+1, :]) * hop_sec  # (N,K)

    pos_hat = np.zeros((K, 2))
    info_list = []
    for i in range(K):
        s_i, info_i = tdoa_gn(P, dt_base[:, i], c, ref=ref)
        pos_hat[i] = s_i
        info_list.append(info_i)

    pos_samps = np.zeros((B, K, 2))
    dt_samps  = np.zeros((B, N, K))
    base = reconstruct_from_params(W, F, S, tau_f, tau_t) if base_X is None else base_X
    for b in range(B):
        Xb = add_noise_by_snr_clipped_per_sensor(base, snr_db=snr_db, rng=rng)
        tau_b = reestimate_tau_t_shifts_only(Xb, W, F, S, tau_f, tau_t, iters_w=20, iters_snap=20)
        tau_unw_b = unwrap_frames(tau_b, Tframes)
        dt_b = (tau_unw_b - tau_unw_b[ref:ref+1, :]) * hop_sec   # (N,K)
        dt_samps[b] = dt_b
        for i in range(K):
            s_b, _ = tdoa_gn(P, dt_b[:, i], c, ref=ref, x0=pos_hat[i])
            pos_samps[b, i] = s_b

    covs = [np.cov(pos_samps[:, i, :].T) for i in range(K)]
    extras = {"dt_base": dt_base, "dt_samples": dt_samps, "pos_samples": pos_samps, "info": info_list}
    return pos_hat, covs, extras 


def tdoa_frames_from_positions(P, Spos, c, hop_sec, ref=0):
    N_ = P.shape[0] 
    K_ = Spos.shape[0]
    tau = np.zeros((N_, K_))
    for i in range(K_):
        d = np.linalg.norm(P - Spos[i], axis=1)
        dt = (d - d[ref]) / c
        tau[:, i] = dt / hop_sec
    return tau

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import pandas as pd

# ---- Parameters & geometry
N, Fdim, Tdim, K = 3, 96, 64, 2
SENSOR_POS = np.array([[0.0, 0.0],[2.0, 0.0],[0.6, 1.8]], dtype=float)
TRUE_SRC_POS = np.array([[1.1, 0.9],[1.7, 0.3]], dtype=float)
HOP_SEC = 0.01
C_PROP = 343.0
BIN_HZ = 1.0

# ---- Synthetic data
rng = np.random.default_rng(0)
f = np.arange(Fdim)
t = np.arange(Tdim)
true_F = np.stack([np.exp(-0.5*((f-28)/5)**2), np.exp(-0.5*((f-62)/7)**2)])
true_S = np.stack([np.exp(-0.5*((t-18)/4)**2), np.exp(-0.5*((t-44)/6)**2)])
true_F = true_F / (true_F.sum(axis=1, keepdims=True) + 1e-12)
true_S = true_S / (true_S.sum(axis=1, keepdims=True) + 1e-12)
true_W = rng.random((N, K))
true_tau_f = rng.uniform(-1.5, 1.5, size=(N, K))


true_tau_t = tdoa_frames_from_positions(SENSOR_POS, TRUE_SRC_POS, C_PROP, HOP_SEC, ref=0)

X_clean = reconstruct_from_params(true_W, true_F, true_S, true_tau_f, true_tau_t)
SNR_DB = 30.0
X = add_noise_by_snr_clipped_per_sensor(X_clean, snr_db=SNR_DB, rng=rng)

print("Norms ||X[n]||:", [float(np.linalg.norm(X[n])) for n in range(N)])
print("Mean |tau_t| frames:", float(np.mean(np.abs(true_tau_t))))

# ---- Seed & fit (arrays-only API)
tau_f0, tau_t0 = seed_shifts_xcorr(X, true_F, true_S)
W0 = np.ones((N, K)) / N

# shifts-only
W_shift, F_shift, S_shift, tau_f_shift, tau_t_shift, hist_shift = fit(
    X, K, mode="shifts_only",
    iters_w=120, iters_snap=150,
    snap_window_f=3, snap_window_t=3,
    ridge_w=0.0, verbose=True,
    W0=W0, F0=true_F.copy(), S0=true_S.copy(), tau_f0=tau_f0, tau_t0=tau_t0, seed=0
)

# staged v12
W_staged, F_staged, S_staged, tau_f_staged, tau_t_staged, hist_staged = fit(
    X, K, mode="staged",
    iters_w=60, iters_fs=120, iters_snap=80,
    inner_rank1_iters=1, snap_window_f=3, snap_window_t=3,
    ridge_w=0.0, verbose=True,
    W0=W0, F0=true_F.copy(), S0=true_S.copy(), tau_f0=tau_f0, tau_t0=tau_t0, seed=0
)

print("R (shifts-only):", relerr(reconstruct_from_params(W_shift, F_shift, S_shift, tau_f_shift, tau_t_shift), X))
print("R (staged v12): ", relerr(reconstruct_from_params(W_staged, F_staged, S_staged, tau_f_staged, tau_t_staged), X))

# ---- Triangulation + bootstrap
pos_hat, covs, tri_raw = bootstrap_triangulation(
    W_staged, F_staged, S_staged, tau_f_staged, tau_t_staged,
    SENSOR_POS, HOP_SEC, C_PROP, ref=0, B=200, rng=np.random.default_rng(123),
    snr_db=SNR_DB, base_X=X_clean
)
dt_base = tri_raw["dt_base"]
dt_samps = tri_raw["dt_samples"]
print("Estimated positions (m):\n", np.round(pos_hat,3))
print("True positions (m):\n",      np.round(TRUE_SRC_POS,3))

# ---- CRB using estimated per-sensor TDOA variance
sigma_dt_est = dt_samps.std(axis=(0,2))  # (N,)
print("Estimated σ_dt per sensor (s):", np.round(sigma_dt_est, 6))
covs_crb = [crb_covariance(SENSOR_POS, dt_base[:, i], C_PROP, ref=0,
                            sigma_tdoa_sec=sigma_dt_est, s_eval=pos_hat[i]) for i in range(K)]

# ---- Plot sensors, true & estimated, 2σ bootstrap and CRB ellipses
def cov_ellipse(mu, cov, nsig=2.0, **kw):
    w, v = np.linalg.eigh(cov)
    w = np.clip(w, 0, None)
    order = np.argsort(w)[::-1]
    w, v = w[order], v[:, order]
    width, height = 2*nsig*np.sqrt(w)
    angle = np.degrees(np.arctan2(v[1,0], v[0,0]))
    return Ellipse(xy=mu, width=width, height=height, angle=angle, fill=False, **kw)

fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(SENSOR_POS[:,0], SENSOR_POS[:,1], marker='^', s=100, c="k", label="sensors", zorder=2)
for i in range(N): ax.text(SENSOR_POS[i,0], SENSOR_POS[i,1], f" {i}", va='bottom', fontsize=9)
ax.scatter(TRUE_SRC_POS[:,0], TRUE_SRC_POS[:,1], marker='*', s=160, c="tab:green", label="true src", zorder=3)
ax.scatter(pos_hat[:,0], pos_hat[:,1], facecolors='none', edgecolors="tab:blue",
            marker='o', s=90, linewidths=2, label="est src", zorder=4)

colors = ["tab:blue","tab:orange","tab:red","tab:purple"]
for i in range(K):
    e_boot = cov_ellipse(pos_hat[i], covs[i], nsig=2.0, edgecolor=colors[i%len(colors)], linewidth=2.5, zorder=5, label="bootstrap 2σ" if i==0 else None)
    ax.add_patch(e_boot)
    if covs_crb[i] is not None:
        e_crb = cov_ellipse(pos_hat[i], covs_crb[i], nsig=2.0, edgecolor=colors[i%len(colors)], linestyle='--', linewidth=2.0, zorder=5, label="CRB 2σ" if i==0 else None)
        ax.add_patch(e_crb)
    ax.text(pos_hat[i,0], pos_hat[i,1], f"  src {i}", color=colors[i%len(colors)], zorder=6)

ax.set_aspect('equal', 'box')
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("Triangulation — bootstrap vs CRB (2σ)")
ax.grid(True)
ax.legend(loc="best")
plt.show()

# ---- Table: true vs estimated positions, bootstrap & CRB uncertainties
def ellipse_params(cov, nsig=2.0):
    w, v = np.linalg.eigh(cov)
    idx = np.argsort(w)[::-1]
    w, v = w[idx], v[:, idx]
    a = nsig*np.sqrt(max(w[0], 0.0))
    b = nsig*np.sqrt(max(w[1], 0.0))
    ang = np.degrees(np.arctan2(v[1,0], v[0,0]))
    area = np.pi*a*b
    return a, b, ang, area

rows = []
for i in range(K):
    ab,bb,angb,areab = ellipse_params(covs[i], nsig=2.0)
    if covs_crb[i] is not None:
        acrb,bcrb,angc,areac = ellipse_params(covs_crb[i], nsig=2.0)
    else:
        acrb=bcrb=angc=areac=np.nan
    err = float(np.hypot(pos_hat[i,0]-TRUE_SRC_POS[i,0], pos_hat[i,1]-TRUE_SRC_POS[i,1]))
    rows.append(dict(
        source=i,
        true_x=TRUE_SRC_POS[i,0], true_y=TRUE_SRC_POS[i,1],
        est_x=pos_hat[i,0], est_y=pos_hat[i,1],
        err_euclid=err,
        boot_semimajor_m=ab, boot_semiminor_m=bb, boot_angle_deg=angb, boot_area_m2=areab,
        crb_semimajor_m=acrb, crb_semiminor_m=bcrb, crb_angle_deg=angc, crb_area_m2=areac
    ))
df_pos = pd.DataFrame(rows, columns=[
    "source","true_x","true_y","est_x","est_y","err_euclid",
    "boot_semimajor_m","boot_semiminor_m","boot_angle_deg","boot_area_m2",
    "crb_semimajor_m","crb_semiminor_m","crb_angle_deg","crb_area_m2"
])
print("\nTriangulation — True vs Estimated with Bootstrap & CRB (2σ)")
print(df_pos.to_string(index=False, float_format=lambda x: f"{x:0.4f}"))
df_pos.to_csv("triangulation_bootstrap_vs_crb.csv", index=False)
print("Saved -> triangulation_bootstrap_vs_crb.csv")

# ---- Frequency-only Doppler tables (per sensor and aggregated)
Fbins = F_staged.shape[1]
fd_true = unwrap_bins(true_tau_f, Fbins) * float(BIN_HZ)
fd_est  = unwrap_bins(tau_f_staged, Fbins) * float(BIN_HZ)

B = 200
rng = np.random.default_rng(123)
base = X_clean
fd_samps = np.zeros((B, N, K))
for b in range(B):
    Xb = add_noise_by_snr_clipped_per_sensor(base, snr_db=SNR_DB, rng=rng)
    tauf_b = reestimate_tau_f_shifts_only(Xb, W_staged, F_staged, S_staged, tau_f_staged, tau_t_staged)
    fd_samps[b] = unwrap_bins(tauf_b, Fbins) * float(BIN_HZ)
fd_sigma = fd_samps.std(axis=0)

rows = []
for n in range(N):
    for k in range(K):
        rows.append(dict(sensor=n, source=k, fd_true=fd_true[n,k], fd_est=fd_est[n,k], fd_sigma=fd_sigma[n,k]))
df_fd = pd.DataFrame(rows, columns=["sensor","source","fd_true","fd_est","fd_sigma"])
print("\nDoppler Frequency — True vs Estimated (±1σ)")
print(df_fd.to_string(index=False, float_format=lambda x: f"{x:0.5f}"))

W = W_staged
rows_src = []
fd_mean_est = np.zeros((K,)) 
fd_mean_true = np.zeros((K,))
fd_mean_sigma = np.zeros((K,))
for k in range(K):
    w = W[:,k].copy()
    w = w / (w.sum() + 1e-12)
    fd_mean_true[k] = float(np.sum(w * fd_true[:,k]))
    fd_mean_est[k]  = float(np.sum(w * fd_est[:,k]))
    fd_mean_samples = np.sum(fd_samps[:,:,k] * w[None,:], axis=1)
    fd_mean_sigma[k] = float(fd_mean_samples.std())
    rows_src.append(dict(source=k, fd_true_mean=fd_mean_true[k], fd_est_mean=fd_mean_est[k], fd_sigma_mean=fd_mean_sigma[k]))
df_fd_src = pd.DataFrame(rows_src, columns=["source","fd_true_mean","fd_est_mean","fd_sigma_mean"])

print("\nDoppler Frequency — Aggregated per Source (W-weighted across sensors)")
print(df_fd_src.to_string(index=False, float_format=lambda x: f"{x:0.5f}"))

df_fd.to_csv("doppler_frequency_per_sensor.csv", index=False)
df_fd_src.to_csv("doppler_frequency_per_source.csv", index=False)
print("Saved -> doppler_frequency_per_sensor.csv and doppler_frequency_per_source.csv")